In [ ]:
import os
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import tensorly as tl
import dask.array as da
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import yeojohnson
import umap

In [ ]:

# --- Load all segment embeddings ---
emb_dir = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_per_segment_analysis/embeddings")
#emb_dir = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_per_segment_analysis/embeddings_tuned")
#emb_dir = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_per_segment_analysis/embeddings_tuned_v2")

segment_names = ["PB2", "PB1", "PA", "HA", "NP", "NA", "MP", "NS"]

embeddings = {}
for seg in segment_names:
    f = emb_dir / f"embeddings_{seg}.parquet"
    if f.exists():
        embeddings[seg] = pd.read_parquet(f)
        print(f"{seg}: {len(embeddings[seg])} samples")
    else:
        print(f"{seg}: FILE NOT FOUND")

In [ ]:
# --- Parse metadata from sample_id ---
# Header format: A/H5N0|A/chicken/Fujian/...|PB2|1|EPI_ISL_...|china|asia|2018

def parse_metadata(df):
    split = df["sample_id"].str.split("|")
    df = df.copy()
    df["country"] = split.str[5]
    df["continent"] = split.str[6]
    df["year"] = split.str[7]
    print(split)
    return df

for seg in embeddings:
    embeddings[seg] = parse_metadata(embeddings[seg])
    embeddings[seg]["year"] = pd.to_numeric(embeddings[seg]["year"], errors="coerce")

# Quick check
embeddings[segment_names[0]][["sample_id", "country", "continent", "year"]].head()


In [ ]:
# --- Plot all 4 methods × 8 segments (plain, no coloring) ---
methods = [("umap_1", "umap_2", "UMAP"),
           ("tsne_1", "tsne_2", "t-SNE"),
           ("mds_1",  "mds_2",  "MDS"),
           ("phate_1","phate_2","PHATE")]

fig, axes = plt.subplots(len(methods), len(segment_names), figsize=(4 * len(segment_names), 4 * len(methods)))

for row, (x_col, y_col, method_name) in enumerate(methods):
    for col, seg in enumerate(segment_names):
        ax = axes[row, col]
        df = embeddings[seg]
        ax.scatter(df[x_col], df[y_col], s=1, alpha=0.3, rasterized=True)
        if row == 0:
            ax.set_title(seg, fontsize=25, fontweight="bold")
        if col == 0:
            ax.set_ylabel(method_name, fontsize=25, fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("Embeddings per segment (all methods)", fontsize=40, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# --- Color by continent ---
def plot_embeddings_colored(embeddings, segment_names, color_col, title, cmap="tab10", categorical=True):
    methods = [("umap_1", "umap_2", "UMAP"),
               ("tsne_1", "tsne_2", "t-SNE"),
               ("mds_1",  "mds_2",  "MDS"),
               ("phate_1","phate_2","PHATE")]
    
    fig, axes = plt.subplots(len(methods), len(segment_names),
                             figsize=(4 * len(segment_names), 4 * len(methods)))
    
    if categorical:
        # Build a shared color map across all segments
        all_vals = pd.concat([embeddings[s][color_col] for s in segment_names]).unique()
        all_vals = sorted([v for v in all_vals if pd.notna(v)])
        color_map = {v: plt.get_cmap(cmap)(i / max(len(all_vals) - 1, 1)) for i, v in enumerate(all_vals)}
    
    for row, (x_col, y_col, method_name) in enumerate(methods):
        for col, seg in enumerate(segment_names):
            ax = axes[row, col]
            df = embeddings[seg]
            
            if categorical:
                colors = df[color_col].map(color_map)
                ax.scatter(df[x_col], df[y_col], c=colors, s=1, alpha=0.3, rasterized=True)
            else:
                sc = ax.scatter(df[x_col], df[y_col], c=df[color_col].astype(float),
                                s=1, alpha=0.3, cmap=cmap, rasterized=True)
            
            if row == 0:
                ax.set_title(seg, fontsize=25, fontweight="bold")
            if col == 0:
                ax.set_ylabel(method_name, fontsize=25, fontweight="bold")
            ax.set_xticks([])
            ax.set_yticks([])
    
    # Legend / colorbar
    if categorical:
        from matplotlib.lines import Line2D
        handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[v],
                          markersize=20, label=v) for v in all_vals]
        fig.legend(handles=handles, loc="center right", title=color_col.capitalize(),
                   fontsize=25, bbox_to_anchor=(1.10, 0.5), title_fontsize=30)
    else:
        cbar_ax = fig.add_axes([1.02, 0.25, 0.02, 0.5])
        cbar = fig.colorbar(sc, ax=axes, shrink=0.5, label=color_col.capitalize(),
                    pad=0.02,        # distance from the plot (default ~0.05)
                    aspect=30,       # height-to-width ratio
                    fraction=0.05,
                    cax=cbar_ax)
       # cbar.set_label(color_col.capitalize(), fontsize=25)  # label fontsize
        cbar.ax.set_title(color_col.capitalize(), fontsize=35, pad=15)
        cbar.ax.tick_params(labelsize=18) 

    
    plt.suptitle(f"{title}", fontsize=40, y=1.00)
    plt.tight_layout()
    plt.show()

In [ ]:
# Color by continent
plot_embeddings_colored(embeddings, segment_names, "continent", "Embeddings Colored by Continent")

In [ ]:
# Color by year (numeric colormap)
plot_embeddings_colored(embeddings, segment_names, "year", "Embeddings Colored by Year",
                        cmap="viridis", categorical=False)

In [ ]:
# Color by country (may have many categories — consider filtering to top N)
top_countries = pd.concat([embeddings[s]["country"] for s in segment_names]).value_counts().head(15).index
emb_filtered = {s: embeddings[s][embeddings[s]["country"].isin(top_countries)] for s in segment_names}
plot_embeddings_colored(emb_filtered, segment_names, "country",
                        f"Embeddings colored by country (top {len(top_countries)})", cmap="tab20")